In [1]:
from supabase import create_client, Client
import os
from dotenv import load_dotenv
import random
import os
import psycopg
import math
import os

import mlflow


from dotenv import load_dotenv
from google.cloud import storage
from mlflow.tracking import MlflowClient
import torch
import torch.nn as nn
import torch.optim as optim
from safetensors.torch import save_file

load_dotenv(override=True)

True

In [4]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "service_account.json"
gcs_bucket = os.getenv("GCS_BUCKET_NAME")
tracking_uri = os.getenv("DATABASE_LOGS_URL")

# Make sure storage exists
client = storage.Client()
bucket = client.bucket(gcs_bucket)
print(f"Bucket exists: {bucket.exists()}")

# Set artifact uri
artifact_uri = f"gs://{gcs_bucket}/mlflow-artifacts"
os.environ["MLFLOW_ARTIFACT_ROOT"] = artifact_uri
mlflow.set_tracking_uri(tracking_uri)

# Set Experiment
experiment_name = "train_ml-2"

try:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=artifact_uri
    )
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(
    experiment_name
)

Bucket exists: True


<Experiment: artifact_location='gs://test-training-ml/mlflow-artifacts', creation_time=1779627200617, experiment_id='7', last_update_time=1779627200617, lifecycle_stage='active', name='train_ml-2', tags={}, trace_location=None, workspace='default'>

In [5]:
# Model
model = nn.Linear(1, 1)

# Dataset
X = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]])
y = torch.tensor([[3.0], [5.0], [7.0], [9.0], [11.0]])

# Optimizer and Loss Function
criterion = nn.MSELoss()

learning_rate = 0.01
epochs = 1

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# Start experiment
with mlflow.start_run():

    # Log parameters
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("epochs", epochs)

    for epoch in range(epochs):
        # Forward pass
        predictions = model(X)
        loss = criterion(predictions, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Simple accuracy calculation
        with torch.no_grad():
            mae = torch.mean(torch.abs(predictions - y))
            accuracy = 1 / (1 + mae.item())

        # Log metrics
        mlflow.log_metric("loss", loss.item(), step=epoch)
        mlflow.log_metric("accuracy", accuracy, step=epoch)

        if epoch % 10 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}")

    # Save model as artifact
    model_path = "model.safetensors"
    save_file(model.state_dict(), model_path)
    mlflow.log_artifact(model_path)

    # Remove local model file after logging
    os.remove(model_path)

Epoch 0, Loss: 52.6815, Accuracy: 0.1305
